In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the collected Bitcoin news dataset
news_path = Path("../data/news/raw/bitcoin_news_72h.csv")
news_df = pd.read_csv(news_path)

print("========== Dataset Overview ==========")
print("File:", news_path.resolve())
print("Total articles:", len(news_df))
print("Columns:", news_df.columns.tolist())

print("\n========== First 5 Articles ==========")
display(news_df.head())

========== Dataset Overview ==========
File: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\news\raw\bitcoin_news_72h.csv
Total articles: 198
Columns: ['title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'content_status']

========== First 5 Articles ==========


,title,source,published_at_utc,link,summary,content,content_status
0,Brazilian companies boosted stablecoins in wee...,Yahoo Finance Video,2026-10-28T19:00:00+00:00,https://finance.yahoo.com/video/brazilian-comp...,Scott Melker discusses how Brazilian companies...,Scott Melker discusses how Brazilian companies...,summary_and_content
1,Bitwise Projects $1.6 Million Bitcoin by 2036....,BeInCrypto,2026-10-29T05:42:09+00:00,https://finance.yahoo.com/markets/crypto/artic...,Bitwise’s $1.6 million Bitcoin target draws ab...,Bitwise projects Bitcoin (BTC) will hit $1.6 m...,summary_and_content
2,Treasury Buyback Fills $6 Billion Again: So Wh...,BeInCrypto,2026-10-29T02:37:44+00:00,https://finance.yahoo.com/markets/crypto/artic...,"Treasury filled its $6 billion buyback again, ...",A US Treasury buyback filled its full $6 billi...,summary_and_content
3,Bitcoin Controls 59.5% of the Crypto Market. I...,24/7 Wall St.,2026-10-29T01:41:42+00:00,https://finance.yahoo.com/markets/crypto/artic...,Bitcoin now commands nearly 60% of the entire ...,Bitcoin's 59.5% crypto market dominance reflec...,summary_and_content
4,What Is Going On With MARA Stock After Its Ste...,Trefis,2026-10-29T00:14:30+00:00,https://finance.yahoo.com/markets/stocks/artic...,For questions about this article or underlying...,For questions about this article or underlying...,summary_and_content


In [3]:
# Check for duplicate records and missing values

print("========== 1. Duplicate Check ==========")
print("Duplicate links:", news_df["link"].duplicated().sum())
print("Duplicate titles:", news_df["title"].duplicated().sum())

print("\n========== 2. Missing Values ==========")
print(news_df[
    ["title", "source", "published_at_utc",
     "link", "summary", "content", "content_status"]
].isna().sum())

print("\n========== 3. Empty Text Check ==========")
print("Empty titles:",
      news_df["title"].fillna("").str.strip().eq("").sum())
print("Empty links:",
      news_df["link"].fillna("").str.strip().eq("").sum())
print("Empty content:",
      news_df["content"].fillna("").str.strip().eq("").sum())

print("\n========== 4. Content Status ==========")
print(news_df["content_status"].value_counts(dropna=False))

print("\n========== 5. Content Length ==========")
content_length = news_df["content"].fillna("").str.len()
print("Minimum content length:", content_length.min())
print("Content under 500 characters:", (content_length < 500).sum())

print("\n========== 6. Summary Check ==========")
summary = news_df["summary"].fillna("").str.strip()
content = news_df["content"].fillna("").str.strip()

print("Summary is None/empty:", summary.eq("").sum())
print("Summary identical to content:", (summary == content).sum())

========== 1. Duplicate Check ==========
Duplicate links: 0
Duplicate titles: 3

========== 2. Missing Values ==========
title               0
source              0
published_at_utc    0
link                0
summary             9
content             0
content_status      0
dtype: int64

========== 3. Empty Text Check ==========
Empty titles: 0
Empty links: 0
Empty content: 0

========== 4. Content Status ==========
content_status
summary_and_content    189
content_only             9
Name: count, dtype: int64

========== 5. Content Length ==========
Minimum content length: 100
Content under 500 characters: 9

========== 6. Summary Check ==========
Summary is None/empty: 9
Summary identical to content: 0


In [4]:
# Check publication timestamps against the latest 72-hour window

from datetime import datetime, timezone

check_time = datetime.now(timezone.utc)
cutoff_time = pd.Timestamp(check_time) - pd.Timedelta(hours=72)

news_df["published_at_utc"] = pd.to_datetime(
    news_df["published_at_utc"],
    utc=True,
    errors="coerce"
)

invalid_timestamps = news_df["published_at_utc"].isna()

outside_window = (
    (news_df["published_at_utc"] < cutoff_time) |
    (news_df["published_at_utc"] > pd.Timestamp(check_time))
) & ~invalid_timestamps

print("========== Publication Time Check ==========")
print("Check time (UTC):", check_time.isoformat())
print("72-hour cutoff (UTC):", cutoff_time.isoformat())
print("Invalid timestamps:", invalid_timestamps.sum())
print("Articles outside 72-hour window:", outside_window.sum())

valid_times = news_df.loc[
    ~invalid_timestamps, "published_at_utc"
]

if not valid_times.empty:
    print("Newest article (UTC):", valid_times.max())
    print("Oldest article (UTC):", valid_times.min())

if outside_window.any():
    print("\nArticles outside the 72-hour window:")
    display(
        news_df.loc[
            outside_window,
            ["title", "published_at_utc", "link"]
        ]
    )

# Filter articles within the latest 72 hours
recent_news_df = news_df.loc[
    news_df["published_at_utc"].between(
        cutoff_time,
        pd.Timestamp(check_time),
        inclusive="both"
    )
].copy()

print("Articles within the latest 72 hours:", len(recent_news_df))

========== Publication Time Check ==========
Check time (UTC): 2026-10-29T06:25:43.035405+00:00
72-hour cutoff (UTC): 2026-10-26T06:25:43.035405+00:00
Invalid timestamps: 0
Articles outside 72-hour window: 0
Newest article (UTC): 2026-10-29 05:42:09+00:00
Oldest article (UTC): 2026-10-26 06:34:10+00:00
Articles within the latest 72 hours: 198


In [5]:
# Identify articles that may be unrelated to Bitcoin or cryptocurrency

import re

crypto_keywords = [
    "bitcoin", "btc", "cryptocurrency", "crypto",
    "blockchain", "digital asset", "digital assets",
    "bitcoin etf", "crypto etf"
]

def check_crypto_relevance(row):
    text = f"{row['title']} {row['summary']} {row['content']}"
    text = text.lower()
    return any(
        re.search(r"\b" + re.escape(keyword) + r"\b", text)
        for keyword in crypto_keywords
    )

news_df["keyword_relevant"] = news_df.apply(
    check_crypto_relevance, axis=1
)

potentially_irrelevant = news_df[
    ~news_df["keyword_relevant"]
]

print("Total articles:", len(news_df))
print("Potentially relevant:", news_df["keyword_relevant"].sum())
print("Potentially irrelevant:", len(potentially_irrelevant))

display(
    potentially_irrelevant[
        ["title", "source", "summary"]
    ]
)

Total articles: 198
Potentially relevant: 190
Potentially irrelevant: 8


,title,source,summary
21,Sector Update: Financial Stocks Rise Late Afte...,MT Newswires,NaN
26,Sector Update: Financial Stocks Rise Wednesday...,MT Newswires,NaN
78,Sector Update: Financial Stocks Decline Late A...,MT Newswires,NaN
83,Sector Update: Financial Stocks Edge Lower Tue...,MT Newswires,NaN
116,Top Premarket Decliners,MT Newswires,NaN
128,Sector Update: Financial Stocks Mixed Late Aft...,MT Newswires,NaN
133,Sector Update: Financial Stocks Advance Monday...,MT Newswires,NaN
158,"Exchange-Traded Funds, Equity Futures Higher P...",MT Newswires,NaN


In [6]:
# Inspect potentially irrelevant articles before making any decisions

review_columns = ["title", "source", "link", "content"]

for index, row in potentially_irrelevant.iterrows():
    print("\n" + "=" * 100)
    print(f"Index: {index}")
    print(f"Title: {row['title']}")
    print(f"Source: {row['source']}")
    print(f"Link: {row['link']}")
    print(f"Content length: {len(str(row['content']))}")
    print("\nFull content:")
    print(row["content"])


Index: 21
Title: Sector Update: Financial Stocks Rise Late Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-rise-afternoon-195617586.html
Content length: 100

Full content:
Financial stocks rose in late Wednesday afternoon trading with the NYSE Financial Index gaining 0.1%

Index: 26
Title: Sector Update: Financial Stocks Rise Wednesday Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-rise-wednesday-181331650.html
Content length: 100

Full content:
Financial stocks were advancing in Wednesday afternoon trading, with the NYSE Financial Index up 0.2

Index: 78
Title: Sector Update: Financial Stocks Decline Late Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-decline-afternoon-195607896.html
Content length: 100

Full content:
Financial stocks were lower in late Tuesday afternoon trading, with the NYSE Fi

In [7]:
# Keyword-based filtering after the 72-hour date filter
crypto_keywords = [
    "bitcoin",
    "btc",
    "cryptocurrency",
    "cryptocurrencies",
    "crypto",
    "blockchain",
    "digital asset",
    "digital assets"
]

def is_crypto_related(row):
    fields = [
        row.get("title", ""),
        row.get("summary", ""),
        row.get("content", "")
    ]

    text = " ".join(
        "" if pd.isna(value) else str(value)
        for value in fields
    ).lower()

    return any(
        re.search(
            r"\b" + re.escape(keyword) + r"\b",
            text
        )
        for keyword in crypto_keywords
    )


# Keep the original datasets unchanged
keyword_mask = recent_news_df.apply(
    is_crypto_related, axis=1
)

keyword_filtered_df = recent_news_df.loc[
    keyword_mask
].copy()

excluded_news_df = recent_news_df.loc[
    ~keyword_mask
].copy()

print("========== Keyword Filtering Summary ==========")
print("Articles after date filtering:", len(recent_news_df))
print("Keyword-relevant articles:", len(keyword_filtered_df))
print("Articles excluded by keyword filter:", len(excluded_news_df))

print("\n========== Excluded Articles ==========")
display(
    excluded_news_df[
        ["title", "source", "published_at_utc"]
    ]
)


========== Keyword Filtering Summary ==========
Articles after date filtering: 198
Keyword-relevant articles: 190
Articles excluded by keyword filter: 8

========== Excluded Articles ==========


,title,source,published_at_utc
21,Sector Update: Financial Stocks Rise Late Afte...,MT Newswires,2026-10-28 19:56:17+00:00
26,Sector Update: Financial Stocks Rise Wednesday...,MT Newswires,2026-10-28 18:13:31+00:00
78,Sector Update: Financial Stocks Decline Late A...,MT Newswires,2026-10-27 19:56:07+00:00
83,Sector Update: Financial Stocks Edge Lower Tue...,MT Newswires,2026-10-27 18:00:47+00:00
116,Top Premarket Decliners,MT Newswires,2026-10-27 11:40:27+00:00
128,Sector Update: Financial Stocks Mixed Late Aft...,MT Newswires,2026-10-26 19:54:03+00:00
133,Sector Update: Financial Stocks Advance Monday...,MT Newswires,2026-10-26 18:22:00+00:00
158,"Exchange-Traded Funds, Equity Futures Higher P...",MT Newswires,2026-10-26 13:18:10+00:00


In [8]:
print("Date-filtered articles:", len(recent_news_df))
print("Keyword-filtered articles:", len(keyword_filtered_df))
print("Excluded articles:", len(excluded_news_df))

print("\nRetained article sample:")
display(
    keyword_filtered_df[
        ["title", "source", "published_at_utc"]
    ].head(5)
)

Date-filtered articles: 198
Keyword-filtered articles: 190
Excluded articles: 8

Retained article sample:


,title,source,published_at_utc
0,Brazilian companies boosted stablecoins in wee...,Yahoo Finance Video,2026-10-28 19:00:00+00:00
1,Bitwise Projects $1.6 Million Bitcoin by 2036....,BeInCrypto,2026-10-29 05:42:09+00:00
2,Treasury Buyback Fills $6 Billion Again: So Wh...,BeInCrypto,2026-10-29 02:37:44+00:00
3,Bitcoin Controls 59.5% of the Crypto Market. I...,24/7 Wall St.,2026-10-29 01:41:42+00:00
4,What Is Going On With MARA Stock After Its Ste...,Trefis,2026-10-29 00:14:30+00:00


## SBERT

In [9]:
# Load the SBERT model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully.")
print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())

INFO: No device provided, using cpu
INFO: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fc

SBERT model loaded successfully.
Embedding dimension: 384


C:\Users\nyy32\AppData\Local\Temp\ipykernel_1244\4262035451.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())


In [10]:
# Define a neutral forecasting query
forecasting_query = """
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.
"""

# Prepare news text for SBERT
def build_article_text(row):
    text_parts = []

    for column in ["title", "summary", "content"]:
        value = row.get(column)

        if pd.notna(value) and str(value).strip():
            text_parts.append(
                f"{column.capitalize()}: {str(value).strip()}"
            )

    return "\n".join(text_parts)


news_texts = [
    build_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Query and Text Preparation ==========")
print("Forecasting query prepared:", bool(forecasting_query.strip()))
print("Candidate articles:", len(news_texts))
print("Empty article texts:", sum(not text.strip() for text in news_texts))

print("\n========== Forecasting Query ==========")
print(forecasting_query.strip())

print("\n========== First Article Text Preview ==========")
print(news_texts[0][:600] if news_texts else "No articles available.")

========== Query and Text Preparation ==========
Forecasting query prepared: True
Candidate articles: 190
Empty article texts: 0

========== Forecasting Query ==========
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.

========== First Article Text Preview ==========
Title: Brazilian companies boosted stablecoins in weekend trading during heated election
Summary: Scott Melker discusses how Brazilian companies boosted stablecoin volumes over the weekend to prevent a currency crash, which occurred at the same time as the highly contentious Brazilian presidential election.

"The Daily Wolf with Scott Melker" airs every day at 12:00 p.m. Tune in for your daily dose of all things crypto.

Make sure to also check out Yahoo Finance's new crypto hub to find t

In [11]:
# 1. Encode the forecasting query
query_embedding = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Encode all candidate news articles
news_embeddings = sbert_model.encode(
    news_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Calculate cosine similarity
similarity_scores = util.cos_sim(
    query_embedding,
    news_embeddings
).cpu().numpy().flatten()

# 4. Create a separate results DataFrame
semantic_results_df = keyword_filtered_df.copy()

semantic_results_df["similarity_score"] = similarity_scores

# 5. Rank articles by semantic similarity
semantic_results_df = (
    semantic_results_df
    .sort_values(
        by="similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

semantic_results_df["similarity_rank"] = (
    range(1, len(semantic_results_df) + 1)
)

# 6. Display the retrieval summary
print("========== SBERT Retrieval Summary ==========")
print("Candidate articles:", len(news_texts))
print("Query embedding dimension:", query_embedding.shape[-1])
print("News embeddings shape:", tuple(news_embeddings.shape))
print("Similarity scores calculated:", len(similarity_scores))

print("\n========== Top 10 Retrieved Articles ==========")

display(
    semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ].head(10)
)

Batches: 100%|██████████| 6/6 [00:03<00:00,  1.67it/s]

========== SBERT Retrieval Summary ==========
Candidate articles: 190
Query embedding dimension: 384
News embeddings shape: (190, 384)
Similarity scores calculated: 190

========== Top 10 Retrieved Articles ==========


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,"Bitcoin Slips Ahead Of Fed Week, Big Tech Earn...",Stocktwits,2026-10-26 10:21:09+00:00,0.622446
1,2,Bitcoin On Track To Post Weakest October In Si...,Stocktwits,2026-10-28 09:22:33+00:00,0.619871
2,3,"Crypto Market Today, Oct. 28: Bitcoin and Altc...",Motley Fool,2026-10-28 21:30:01+00:00,0.602598
3,4,"Bitcoin little-changed at $80.4k as oil, rate ...",Investing.com,2026-10-26 06:54:50+00:00,0.600868
4,5,Bitcoin rises slightly after Fed holds interes...,Investing.com,2026-10-28 06:49:56+00:00,0.592786
5,6,"Crypto Market Today, Oct. 26: Bitcoin Slips Be...",Motley Fool,2026-10-26 21:25:02+00:00,0.589900
6,7,"Bitcoin up as risk sentiment, oil slip offset ...",Investing.com,2026-10-27 06:56:49+00:00,0.588652
7,8,"Bitcoin faces further downside as oil surge, A...",Investing.com,2026-10-27 13:53:17+00:00,0.585149
8,9,"$8 Trillion in Big Tech Earnings, Fed Decision...",CCN,2026-10-26 15:32:18+00:00,0.582564
9,10,"Bitcoin Price Bounces as Nasdaq Soars, KOSPI R...",Cryptonews,2026-10-28 08:36:07+00:00,0.572658


In [12]:
# Select the top 10 retrieved news articles

TOP_K = 10

retrieved_news_df = semantic_results_df.head(TOP_K).copy()

print("Retrieved news prepared successfully.")
print("Selected articles:", len(retrieved_news_df))
print(
    "Similarity score range:",
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

Retrieved news prepared successfully.
Selected articles: 10
Similarity score range: 0.5727 to 0.6224


In [13]:
print("========== Retrieved News Quality Check ==========")

print("Retrieved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)
print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

========== Retrieved News Quality Check ==========
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Scores sorted descending: True


In [14]:
# Review the relevance of the top 10 retrieved news articles

for _, row in retrieved_news_df.iterrows():
    print("\n" + "=" * 90)
    print("Rank:", row["similarity_rank"])
    print("Similarity score:", round(row["similarity_score"], 4))
    print("Title:", row["title"])
    print("Source:", row["source"])
    print("Published (UTC):", row["published_at_utc"])

    summary = row.get("summary")
    content = row.get("content")

    print("\nSummary:")
    print(summary if pd.notna(summary) else "None")

    print("\nContent preview:")
    print(str(content)[:600] if pd.notna(content) else "None")


Rank: 1
Similarity score: 0.6224
Title: Bitcoin Slips Ahead Of Fed Week, Big Tech Earnings: Analysts Divided On Where BTC Is Headed Next
Source: Stocktwits
Published (UTC): 2026-10-26 10:21:09+00:00

Summary:
Analyst Michaël van de Poppe pointed to BTC’s 120% annualized return since 2011 against 6% for gold.

Content preview:
Bitcoin slipped below $80,000 on Monday, dipping to around $79,330 in the past day, near its lowest level since mid-September.

Analyst Ted Pillows warned that losing the $80,000 zone on the weekly chart could lead to "capitulation soon."

The Fed decides on rates Wednesday, with Microsoft, Alphabet and Amazon earnings also set to be released this week.

Bitcoin's (BTC) price slipped below $80,000 on Monday ahead of the Federal Reserve's rate decision and earnings from Microsoft (MSFT), Alphabet (GOOG), and Amazon (AMZN) this week, with crypto analysts split on where Bitcoin is headed next.

T

Rank: 2
Similarity score: 0.6199
Title: Bitcoin On Track To Post Weak

In [15]:
# Remove identifiable promotional paragraphs from retrieval text only
ad_patterns = [
    r"Before you buy NVIDIA",
    r"stock picks are beating the market",
    r"See the list now",
    r"See what 10M\+ investors are talking about",
    r"Get the Stocktwits Daily Rip",
    r"Building a portfolio and living off one",
    r"\(Sponsor\)"
]

def build_clean_article_text(row):
    parts = []

    title = row.get("title")
    summary = row.get("summary")
    content = row.get("content")

    if pd.notna(title) and str(title).strip():
        parts.append(f"Title: {str(title).strip()}")

    if pd.notna(summary) and str(summary).strip():
        parts.append(f"Summary: {str(summary).strip()}")

    if pd.notna(content) and str(content).strip():
        paragraphs = re.split(
            r"\n\s*\n",
            str(content).strip()
        )

        clean_paragraphs = [
            paragraph.strip()
            for paragraph in paragraphs
            if paragraph.strip()
            and not any(
                re.search(pattern, paragraph, flags=re.IGNORECASE)
                for pattern in ad_patterns
            )
        ]

        if clean_paragraphs:
            parts.append(
                "Content: " + "\n\n".join(clean_paragraphs)
            )

    return "\n".join(parts)


# Create clean retrieval texts without changing the source DataFrame
news_texts_clean = [
    build_clean_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Retrieval Text Cleaning ==========")
print("Candidate articles:", len(news_texts_clean))
print("Empty article texts:", sum(
    not text.strip() for text in news_texts_clean
))

print("\n========== Cleaned Example ==========")
print(news_texts_clean[2][:1000])

========== Retrieval Text Cleaning ==========
Candidate articles: 190
Empty article texts: 0

========== Cleaned Example ==========
Title: Treasury Buyback Fills $6 Billion Again: So Why Is Bitcoin Below $80K?
Summary: Treasury filled its $6 billion buyback again, but Bitcoin did not rally. Here is what changed since August's jump.
Content: A US Treasury buyback filled its full $6 billion cap in 20- to 30-year bonds on Tuesday. Bitcoin (BTC) did not rally the way it did in August., in fact, it fell below the $80,000 mark.

Bitcoin traders who treated buybacks as a yield catalyst saw no such trigger. August's rally followed a policy surprise, while Tuesday's operation had sat on Treasury's tentative schedule for weeks.

On August 19, Treasury said it would double its long-dated buybacks to at least $4 billion per operation. The announcement caught investors by surprise.

Bitcoin gained about 7% that day, while the 30-year yield pulled back from its highest level in 19 years.

Tuesday br

In [16]:
# 1. Encode the same forecasting query
query_embedding_clean = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Re-encode the cleaned news texts
news_embeddings_clean = sbert_model.encode(
    news_texts_clean,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Recalculate cosine similarity
clean_scores = util.cos_sim(
    query_embedding_clean,
    news_embeddings_clean
).cpu().numpy().flatten()

# 4. Create a new results DataFrame
cleaned_semantic_results_df = keyword_filtered_df.copy()
cleaned_semantic_results_df["similarity_score"] = clean_scores

cleaned_semantic_results_df = (
    cleaned_semantic_results_df
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

cleaned_semantic_results_df["similarity_rank"] = range(
    1, len(cleaned_semantic_results_df) + 1
)

# 5. Compare old and new rankings
rank_comparison = semantic_results_df[
    ["link", "title", "similarity_rank", "similarity_score"]
].merge(
    cleaned_semantic_results_df[
        ["link", "similarity_rank", "similarity_score"]
    ],
    on="link",
    suffixes=("_before", "_after"),
    validate="one_to_one"
)

# 6. Display the new Top 10
print("========== Cleaned SBERT Retrieval Summary ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("New embeddings shape:", tuple(news_embeddings_clean.shape))
print("Similarity scores recalculated:", len(clean_scores))

print("\n========== Top 10 After Cleaning ==========")
display(
    cleaned_semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "similarity_score"
        ]
    ].head(10)
)

print("\n========== Ranking Comparison ==========")
old_top10_links = set(
    semantic_results_df.head(10)["link"]
)
new_top10_links = set(
    cleaned_semantic_results_df.head(10)["link"]
)

print("Top 10 articles retained:", len(old_top10_links & new_top10_links))
print("Articles entering Top 10:", len(new_top10_links - old_top10_links))
print("Articles leaving Top 10:", len(old_top10_links - new_top10_links))

Batches: 100%|██████████| 6/6 [00:03<00:00,  1.61it/s]

========== Cleaned SBERT Retrieval Summary ==========
Candidate articles: 190
New embeddings shape: (190, 384)
Similarity scores recalculated: 190

========== Top 10 After Cleaning ==========


,similarity_rank,title,source,similarity_score
0,1,"Bitcoin Slips Ahead Of Fed Week, Big Tech Earn...",Stocktwits,0.622446
1,2,Bitcoin On Track To Post Weakest October In Si...,Stocktwits,0.619871
2,3,"Crypto Market Today, Oct. 28: Bitcoin and Altc...",Motley Fool,0.602598
3,4,"Bitcoin little-changed at $80.4k as oil, rate ...",Investing.com,0.600868
4,5,Bitcoin rises slightly after Fed holds interes...,Investing.com,0.592786
5,6,"Crypto Market Today, Oct. 26: Bitcoin Slips Be...",Motley Fool,0.589900
6,7,"Is Bitcoin a Good Buy at $80,000 Right Now?",24/7 Wall St.,0.589638
7,8,"Bitcoin up as risk sentiment, oil slip offset ...",Investing.com,0.588652
8,9,"Bitcoin faces further downside as oil surge, A...",Investing.com,0.585149
9,10,"$8 Trillion in Big Tech Earnings, Fed Decision...",CCN,0.582564



========== Ranking Comparison ==========
Top 10 articles retained: 9
Articles entering Top 10: 1
Articles leaving Top 10: 1


In [17]:
# Update the final retrieved news using cleaned SBERT results

TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

print("========== Final Retrieved News ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))
print(
    "Similarity scores descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

display(
    retrieved_news_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ]
)

========== Final Retrieved News ==========
Candidate articles: 190
Retrieved articles: 10
Similarity scores descending: True


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,"Bitcoin Slips Ahead Of Fed Week, Big Tech Earn...",Stocktwits,2026-10-26 10:21:09+00:00,0.622446
1,2,Bitcoin On Track To Post Weakest October In Si...,Stocktwits,2026-10-28 09:22:33+00:00,0.619871
2,3,"Crypto Market Today, Oct. 28: Bitcoin and Altc...",Motley Fool,2026-10-28 21:30:01+00:00,0.602598
3,4,"Bitcoin little-changed at $80.4k as oil, rate ...",Investing.com,2026-10-26 06:54:50+00:00,0.600868
4,5,Bitcoin rises slightly after Fed holds interes...,Investing.com,2026-10-28 06:49:56+00:00,0.592786
5,6,"Crypto Market Today, Oct. 26: Bitcoin Slips Be...",Motley Fool,2026-10-26 21:25:02+00:00,0.589900
6,7,"Is Bitcoin a Good Buy at $80,000 Right Now?",24/7 Wall St.,2026-10-26 14:56:20+00:00,0.589638
7,8,"Bitcoin up as risk sentiment, oil slip offset ...",Investing.com,2026-10-27 06:56:49+00:00,0.588652
8,9,"Bitcoin faces further downside as oil surge, A...",Investing.com,2026-10-27 13:53:17+00:00,0.585149
9,10,"$8 Trillion in Big Tech Earnings, Fed Decision...",CCN,2026-10-26 15:32:18+00:00,0.582564


In [18]:
print("========== Final SBERT Retrieval Quality Check ==========")

print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))

print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)

print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)

print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)

print(
    "Missing content:",
    retrieved_news_df["content"].isna().sum()
)

print(
    "Missing similarity scores:",
    retrieved_news_df["similarity_score"].isna().sum()
)

print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

print(
    "All scores finite:",
    np.isfinite(retrieved_news_df["similarity_score"]).all()
)

print("\nSimilarity score range:")
print(
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

========== Final SBERT Retrieval Quality Check ==========
Candidate articles: 190
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Missing content: 0
Missing similarity scores: 0
Scores sorted descending: True
All scores finite: True

Similarity score range:
0.5826 to 0.6224


In [19]:
# Map cleaned article texts back to their original article links
clean_text_by_link = dict(
    zip(keyword_filtered_df["link"], news_texts_clean)
)

cleaned_semantic_results_df["retrieval_text"] = (
    cleaned_semantic_results_df["link"].map(clean_text_by_link)
)

# Refresh the final Top 10 using cleaned SBERT rankings
TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

# Select the fields needed for downstream interpretation
columns_to_save = [
    "similarity_rank",
    "similarity_score",
    "title",
    "source",
    "published_at_utc",
    "link",
    "summary",
    "content",
    "retrieval_text",
    "content_status"
]

# Save to the project-level processed news directory
output_dir = Path("../data/news/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bitcoin_news_retrieved_top10.csv"

retrieved_news_df[columns_to_save].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Retrieved news saved successfully.")
print("File:", output_path)
print("Saved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)
print("Columns:", columns_to_save)

Retrieved news saved successfully.
File: ..\data\news\processed\bitcoin_news_retrieved_top10.csv
Saved articles: 10
Duplicate links: 0
Scores sorted descending: True
Columns: ['similarity_rank', 'similarity_score', 'title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'retrieval_text', 'content_status']
